# Text clip cosine similarity example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase, SimpleGammaCurve
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.05
n_iter = 200

In [ ]:
global_seed = 2 # Can be None

In [ ]:
import open_clip
clip_model_name = 'ViT-B-16-SigLIP-512'
clip_pretrained = 'webli'
# clip_model_name = 'ViT-L-14-quickgelu'
# clip_pretrained = 'openai'
model, _, preprocess_eval = open_clip.create_model_and_transforms(clip_model_name, pretrained=clip_pretrained, device=device)
tokenizer = open_clip.get_tokenizer(clip_model_name)

# Can specify weights of a fine-tuned model here, so long as the architecture is the same
fine_tune = ''  # TODO: PATH_UPDATE fine-tuned checkpoint path
if fine_tune:
    model.load_state_dict(torch.load(fine_tune, map_location=device))
    print("Loaded fine-tuned weights from ", fine_tune)
model.eval()
print("Model loaded! :)")

In [ ]:
from scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, EinarScene
scene = CarScene()
color_space_converter = SimpleGammaCurve() # LinearRec709ToAgXBase(AgXPunchyLook())

In [ ]:
from losses.clip import CLIPCosineSimilarity, CLIPDirectionalCosineSimilarity

initial_prompt = 'a 3D rendering of a car with boring, ugly, overexposed, flat lighting'
target_prompt = 'a 3D rendering of a car at night'
# criterion = CLIPCosineSimilarity(target_prompt, model, tokenizer, device, preprocess)
criterion = CLIPDirectionalCosineSimilarity(initial_prompt, target_prompt, scene.get_combined_image(color_space_converter).permute(2, 1, 0), model, tokenizer, device=device, preprocess=preprocess_eval, always_prenormalize_vectors=True)
# from losses.image_image import ImageImageCLIPLoss
title_prefix = "ImageTextFineTuning Comparison"

In [ ]:
from utils.train import train_with_criterion
from torchvision.transforms.v2 import RandomChoice, RandomPerspective, RandomResizedCrop, RandomHorizontalFlip, GaussianBlur, Identity, Transform


size = model.visual.preprocess_cfg['size'] or (224, 224)

train_with_criterion(
    scene,
    lr, n_iter, criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="text_clip_cosine_similarity_example",
    n_results=4,
    torch_precision=torch_precision,
    # augmentation=RandomChoice([RandomResizedCrop(size=size, scale=(0.1, 1.0), antialias=True)]),
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix=title_prefix + "Fine-Tuned Model" if fine_tune else "Original Model",
    device=device,
    save_every=40,
    model_name=clip_model_name,
    pretrained_source=fine_tune,
    seed=global_seed,
    show_images_after_augmentation=False
)